In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as sps

In [ ]:
# Download walkscore values 
from bs4 import BeautifulSoup
from urllib.request import urlretrieve, urlopen
import xml.etree.ElementTree as ET
import pandas as pd
import requests

In [ ]:
url = "https://www.walkscore.com/AU-VIC/Melbourne"

response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

walk_score = []
for q in soup.find_all('tr'):
    line = str(q.text)

    if line[1] == 'R':
        continue

    line = line.split("-")[0].split()[1:]
    score = int(line[-1])
    town = " ".join(line[:-1])
    walk_score.append([town, score])

walk_score_df = pd.DataFrame(walk_score, columns=['town', 'walkability'])
walk_score_df.head()

In [ ]:
data = pd.read_csv('../data/curated/merged_data.csv')
postcode_to_town = pd.read_csv('../data/curated/postcode_to_town.csv', index_col=0)
data['house'] = np.where(data['property type'] == 'House', 1, 0)
data['apartment'] = np.where(data['property type'] == 'Apartment / Unit / Flat', 1, 0)
data['townhouse'] = np.where(data['property type'] == 'Townhouse', 1, 0)


In [ ]:
gb = data.groupby(['postcode']).agg(
    avg_beds=('beds', 'mean'),
    avg_baths=('baths', 'mean'),
    avg_parking=('baths', 'mean'),
    avg_rent=('weekly rent amount', 'mean'),
    good_schools=('is closest school good', 'mean'),
    median_rent=('weekly rent amount', 'median'),
    house_percent=('house', 'mean'),
    apartment_percent=('apartment', 'mean'),
    townhouse_percent=('townhouse', 'mean'),
    avg_prmy_school_dist=('distance primary school', 'mean'),
    avg_high_school_dist=('distance secondary school', 'mean'),
    avg_uni_dist=('distance university campus', 'mean'),
    offence_avg=('offence count 2023', 'mean'),
    employed_percent=('employment ratio 2021', 'mean'),
    avg_hospital_dist=('distance to closest hospital', 'mean'),
    population=('estimated resident population (no.) 2021', 'mean'),
    supermarkets=('total_supermarkets', 'mean'),
    coastal=('coastal', 'first'),
    avg_dist_to_city_centre=('closest_cc_dist', 'mean'),
    avg_dist_to_melb=('dist_to_melb', 'mean'),
)

# Now we want to normalise for population
gb['offence_avg'] = gb['offence_avg']
gb['supermarkets'] = gb['supermarkets'] / gb['population']


gb = pd.merge(gb, postcode_to_town, how='left', on='postcode')
gb = pd.merge(gb, walk_score_df, how='left', on='town')

gb_postcode_to_town = postcode_to_town.groupby('postcode')['town'].apply(list)

dct = {}
lst = []

for idx, (postcode, walk) in gb[['postcode', 'walkability']].iterrows():
    dct[postcode] = walk

for idx, (postcode, walk) in gb[['postcode', 'walkability']].iterrows():
    if np.isnan(walk):
        lst.append(dct.get(postcode))
    else:
        lst.append(walk)

gb['walkability'] = lst
gb['walkability'].fillna(np.mean(gb['walkability']), inplace=True)
gb['supermarkets'].fillna(0, inplace=True)

gb['avg_uni_dist'] = np.log(gb['avg_uni_dist'].astype(np.float64) + 0.001)
gb['avg_dist_to_city_centre'] = np.log(gb['avg_dist_to_city_centre'].astype(np.float64) + 1)
gb['avg_dist_to_melb'] = np.log(gb['avg_dist_to_melb'].astype(np.float64) + 1)
gb['avg_hospital_dist'] = np.log(gb['avg_hospital_dist'].astype(np.float64) + 0.001)
gb['avg_prmy_school_dist'] = np.log(gb['avg_prmy_school_dist'].astype(np.float64) + 0.001)
gb['avg_high_school_dist'] = np.log(gb['avg_high_school_dist'].astype(np.float64) + 0.001)

gb.index = gb['postcode']
gb = gb.drop(columns=['town', 'postcode'])
normalised_gb = (gb-gb.min()) / (gb.max()-gb.min())


In [ ]:
ldf = pd.DataFrame()
ldf['spaceScore'] = (normalised_gb['avg_beds'] + 0.4*normalised_gb['avg_baths'] + 0.2*normalised_gb['avg_parking']) / 1.6
ldf['schoolScore'] = (normalised_gb['good_schools'] + (1 - normalised_gb['avg_prmy_school_dist']) + (1 - normalised_gb['avg_high_school_dist'])) / 3
ldf['uniScore'] = (1 - normalised_gb['avg_uni_dist'])
ldf['affordabilityScore'] = (1 - normalised_gb['median_rent']) + (1 - normalised_gb['avg_rent']) / 2 
ldf['houseScore'] = normalised_gb['house_percent']
ldf['apartmentScore'] = normalised_gb['apartment_percent']
ldf['townhouseScore'] = normalised_gb['townhouse_percent']
ldf['home_typeScore'] = 2*normalised_gb['house_percent'] + normalised_gb['townhouse_percent'] + 0.5*normalised_gb['apartment_percent']
ldf['safetyScore'] = 1 - normalised_gb['offence_avg']
ldf['employmentScore'] = normalised_gb['employed_percent']
ldf['beachScore'] = normalised_gb['coastal']
ldf['travelScore'] = 1 - normalised_gb['avg_dist_to_city_centre']
ldf['melbScore'] = 1 - normalised_gb['avg_dist_to_melb']
ldf['walkability'] = normalised_gb['walkability']
ldf['social_infrastrctureScore'] = (1 - normalised_gb['avg_hospital_dist'])

# STILL NEED: GREENSPACES, TRANSPORT, ENTERTAINMENT
# 

ldf = (ldf-ldf.min()) / (ldf.max()-ldf.min())
ldf = ldf.drop_duplicates()

In [ ]:
# Create a baseline score for each suburb

coefficients = [2,2,3,2,3,2,2,3,4,4]
max = sum(coefficients)
x = (ldf['spaceScore'], ldf['home_typeScore'], ldf['schoolScore'], ldf['uniScore'], ldf['safetyScore'], ldf['employmentScore'], ldf['travelScore'], ldf['melbScore'], ldf['walkability'], ldf['social_infrastrctureScore'])
ldf['baseline'] = np.dot(coefficients, x)
ldf['baseline'] = (ldf['baseline'] - ldf['baseline'].mean()) / ldf['baseline'].var()
ldf['baseline'] = 100 * sps.norm.cdf(list(ldf['baseline']))

ranking = pd.merge(ldf, gb_postcode_to_town, on='postcode').sort_values(by='baseline', ascending=False)
ranking = ranking.groupby(['postcode']).agg(baseline=('baseline', 'mean'), town=('town', 'first'))
ranking = ranking.sort_values(by='baseline', ascending=False)


In [ ]:
# How important is safety to you?
# Important - Very important - The most important 
safety = 'Important'
# How important is it to live in a schooling district? 
# Important, want a primary school - Important, want a high school - Important, want both - Don't care - Do not want schools  
school = None
# How important is it to live near a uni?
# Important - Don't care - Do not want unis 
uni = None
# How important is it to live in a walkable suburb?
# Not important at all - Not important - Don't care - Important - The most important
walkability = None
# What type of house do you prefer?
# Apartment - House, don't mind size - Bigger house 
house_size = None
# Are you worried about employability?
# I'm looking for a job - I'm employed, but worried - I'm employed, but secure
employability = None
# Do you want to live by the coast?
# Yes - No
coastal = False
# Do you want to live in a major city centre?
country = None

def preferential_postcode():
    pass